# 00 — Overview

**Goal:** beat the ELO rating at predicting match outcomes. **Data:** a recreational tennis league, 2017–2026, ~18k singles matches, pseudonymized. **Metric:** Brier score (primary); accuracy and AUC are secondary.

## Experiments (single time split, lower Brier is better)

| Exp | Notebook | Idea | Brier (Nov 2026) | Brier (Jul 2026) | ECE p.b. (Nov) | MCB (Nov) | DSC (Nov) |
|---|---|---|---|---|---|---|---|
| ELO | 03 | Closed-form rating formula | 0.1825 | 0.1771 | 5.7 | 0.0060 | 0.0735 |
| A | 03 | ELO-free XGBoost | 0.2048 | 0.1969 | 4.5 | 0.0028 | 0.0480 |
| A′ | 07 | A plus razina trajectory | 0.1830 | 0.1754 | 3.2 | 0.0018 | 0.0688 |
| B | 03 | XGBoost with ELO | 0.1720 | 0.1670 | 2.6 | 0.0017 | 0.0797 |
| B′ | 07 | B plus razina trajectory | 0.1695 | 0.1655 | 2.9 | 0.0015 | 0.0820 |
| C | 05 | ELO-free GRU on sequences | 0.1700 | 0.1628 | 2.4 | 0.0016 | 0.0816 |
| C2 | 08 | GRU with ELO at head | 0.1664 | 0.1609 | 2.6 | 0.0014 | 0.0849 |
| C3 | 10 | GRU with enriched features | 0.1669 | 0.1587 | 3.7 | 0.0024 | 0.0855 |
| C4 | 11 | C3 plus LLM text skill | — | 0.1598 | — | — | — |
| D | 14 | TabPFN v2 in-context (mean of 3 seeds (42, 1, 2), last 10k context; no training) | 0.1688 | — | 2.9 | 0.0013 | 0.0826 |
| D35 | 15 | TabPFN-3.5 in-context (research-only; full context, seed 42; no training) | 0.1666 | — | 1.9 | 0.0011 | 0.0845 |

ECE = mean calibration error in percentage points (10 equal-mass bins). MCB (miscalibration) and DSC (discrimination) come from the CORP decomposition: Brier = MCB − DSC + 0.25.

Both snapshots use the same chronological split rule (last 20% of matches as test). C4 was not re-run on the November snapshot. D is the B′-feature variant (last 10k rows as context). The 0.1688 in the table is the mean of the Brier scores of 3 seeds (42, 1, 2; seed 42 alone is 0.16876); the README reports 0.1686, which is the Brier of the ensemble (mean of the 3 seeds' predictions). It was only run on the November snapshot.

## Walk-forward (4 expanding folds, mean Brier ± std)

| Model | Nov 2026 | Jul 2026 |
|---|---|---|
| ELO | 0.1763 ± 0.0070 | 0.1754 ± 0.0064 |
| B′ | 0.1641 ± 0.0092 | 0.1633 ± 0.0083 |
| C2 | 0.1614 ± 0.0105 | 0.1617 ± 0.0117 |
| C3 | 0.1601 ± 0.0117 | 0.1587 ± 0.0104 |
| D (TabPFN v2, B′ features) | 0.1622 ± 0.0092 | — |
| D35 (TabPFN-3.5, B′ features; research-only) | 0.1609 ± 0.0097 | — |

## Blends (walk-forward, pooled over 4 folds, 14,428 matches, Nov 2026 snapshot)

Fixed 0.5/0.5 weights chosen before looking at results, no fitting on the test data. XGBoost B′ here is the walk-forward run of the same B′ model (pooled Brier 0.16405, 7.0% below ELO). Notebook 16.

| Model | Brier | vs ELO |
|---|---|---|
| 0.5·C3 ensemble + 0.5·TabPFN-3.5 (research-only) | 0.15661 | 11.2% |
| 0.5·C3 ensemble + 0.5·TabPFN v2 | 0.15733 | 10.8% |
| 0.5·C3 ensemble + 0.5·XGBoost B′ | 0.15779 | 10.5% |
| (C3 ensemble + TabPFN v2 + XGBoost B′) / 3 | 0.15784 | 10.5% |
| 0.5·C3 + 0.5·XGBoost B′ | 0.15798 | 10.4% |
| C3 ensemble | 0.15889 | 9.9% |
| C3 | 0.16009 | 9.2% |
| XGBoost B′ | 0.16405 | 7.0% |
| ELO | 0.17633 | — |

**Where does C3 beat ELO?** Mostly discrimination: vs ELO, ΔDSC = +0.0116 [+0.0079, +0.0155] and ΔMCB = −0.0040 [−0.0069, −0.0012] (Nov, paired bootstrap), so about three quarters of the Brier gain is better ranking of matches and one quarter better calibration.

## Figures (Nov 2026 unless noted)

![Brier by model](figures/brier_nov_vs_jul.png)
![Calibration](figures/calibration_nov.png)
![Walk-forward](figures/walkforward_nov.png)
![MCB and DSC](figures/mcb_dsc_nov.png)

## Conclusions

- **C (ELO-free GRU)** is the key research result: sequence modelling beats ELO without ever seeing a rating.
- **C3** is the production model.
- On the November snapshot C3 and C2 are statistically tied (paired bootstrap ΔBrier −0.0005, 95% CI [−0.0031, +0.0021]).
- **Exp D (TabPFN v2)** beats ELO significantly (ΔBrier ≈ −0.014) but not C3 (≈ +0.002; C3 wins 3 of 4 walk-forward folds), so it does not go to production (notebook 14).
- **Exp D35 (TabPFN-3.5, research-only)** reaches the single-seed C3 without training (pooled ΔBrier vs C3 +0.0007, 95% CI [−0.0011, +0.0025]), is better than TabPFN v2 and best calibrated, but the C3 ensemble stays best; never used in production (notebook 15).
- **Blends (notebook 16):** averaging the GRU with a tabular model beats the GRU alone (C3 ensemble + TabPFN v2: ΔBrier −0.00158, 95% CI [−0.00240, −0.00075]; + XGBoost B′: −0.00112 [−0.00203, −0.00023]; the two are statistically indistinguishable: +0.00046 [−0.00012, +0.00104]). The three-way blend does not help (residual correlation 0.95–0.98). The TabPFN-3.5 blend is research-only.

*TabPFN-3.5 (Prior Labs), used under the TabPFN-3.5 Non-Commercial License; research evaluation only, run locally with all outbound network blocked; never used in production — production uses Exp C3. Its numbers are on the November snapshot (split 2025-06-06), not the July one; D35 is a single-seed run (seed 42) on the main split, shown for the B′-feature variant.*

**Reading order:** 01 → 02 → 03 → 04 → 05 → 06 → 07 → 08 → 09 → 10 → 11 → 12 → 13 → 14 → 15 → 16.